# モジュール3 — エージェントループを組み立てる
### RAG + MCPを、判断するAIモデルでつなぐ

モジュール1とモジュール2が、ここでつながります。**エージェントループ**とは、次のような
小さな制御ロジックのことです:
1. ユーザーの質問を見る
2. モデルに判断させる: 直接答えるか、文書を検索するか(モジュール1)、ツールを
   呼ぶか(モジュール2)
3. その判断を実行する
4. 結果を返す

> ⚠️ **RAGを完成させる:** モジュール1で作ったのはRAGの「R」(Retrieval=検索)の部分
> だけでした —— 質問に対して関連チャンクを見つけて、そのまま画面に表示するだけ。
> **このモジュールで、足りなかった「AG」(Augmented Generation=拡張生成)を追加します。**
> チャンクが見つかったら、それを文脈としてモデルにもう一度渡し、実際の回答文を組み立てて
> もらいます。ワークショップの中で初めて、検索して終わりではなく、検索して**生成する**
> という、RAGの完全な形が完成する瞬間です。

> 📌 **今日作る「エージェント」について、正確に理解しておきましょう:**
> - **判断は1回だけです。** 質問を受け取り、1回だけ判断し、1回だけ実行し、1つの答えを
>   返して終わります。結果を振り返って考え直したり、うまくいかなければ別の方法を試したり、
>   複数のツールを連続で呼び出したりはしません。
> - **選べる選択肢は3つだけです。** 「検索する・ツールを呼ぶ・直接答える」のいずれかを
>   選ぶだけで、それ以外のことは何もできません。今日作ったツールも1つだけです。
> - これは「自律的」ではありますが、**あらかじめ用意された選択肢の中で1回だけ判断する**、
>   限定された自律性です。「うまくいくまで何度も試行錯誤する」ような、AutoGPTに代表される
>   タイプのエージェントとは別物です。

**モデルの選択:** ゼロから作ったSLMではなく、小さな**事前学習済みの指示モデル**
(`Qwen2.5-0.5B-Instruct` や `llm-jp-3-150m-instruct3` など)を使います。1回のラボ
セッションでゼロから学習したモデルでは、構造化されたツール呼び出しの出力を安定して
生成することができません —— この「安定性」こそが、このモジュールで学ぶテーマです。
**ローカルで**実行するため(ホスティングされたAPI経由ではないため)、トークンごとの
費用は一切かかりません。

> 💡 **このノートブックは2つのモードで動きます**、下の1つのフラグで切り替えます:
> - `USE_MOCK_LLM = True`(デフォルト): モデルの代わりとなるルールベースの模擬関数。
>   ダウンロードもGPUも不要で、エージェントループ全体が即座に動きます —— まずは
>   **制御の流れ**を理解するのに最適です。
> - `USE_MOCK_LLM = False`: `transformers` 経由で本物の指示モデルをダウンロードして
>   実行します(初回はインターネット接続が必要、約1GBのダウンロード、CPUで1回あたり
>   1〜2分)。

## 環境セットアップ(SageMaker ノートブックインスタンス)

- **インスタンスタイプ:** `ml.t3.medium` で十分です。本物のモデルを使うボーナスセルまで
  試す予定がある場合は `ml.t3.large` の方が安全です。
- **カーネル:** `conda_pytorch_p310`
- **初回のみ:** 下のセルで、このカーネルに入っていないパッケージをインストールします。

In [ ]:
# このノートブックインスタンスで一度だけ実行してください
# conda_pytorch_p310 には numpy/transformers は入っていますが、以下は入っていません:
%pip install --quiet faiss-cpu scikit-learn 'fastmcp>=4.0'

In [ ]:
# セットアップ
import asyncio, json, re
from importlib.metadata import version, PackageNotFoundError
from fastmcp import FastMCP, Client

# MCPの仕様は2026年7月28日に大きな改訂がありました(モジュール2と同じ確認です)
try:
    fastmcp_version = version("fastmcp")
except PackageNotFoundError:
    fastmcp_version = "0.0.0"

major_version = int(fastmcp_version.split(".")[0])
assert major_version >= 4, (
    f"fastmcp {fastmcp_version} が検出されました -- fastmcp>=4.0 が必要です。"
    f"実行してください: pip install --upgrade 'fastmcp>=4.0'"
)

def TODO(hint=""):
    """演習の未完成部分を示す関数です。TODO(...) の呼び出しを自分のコードに置き換えて
    ください。置き換えるまではエラーが出続けます(これは正常な動作です)。"""
    raise NotImplementedError(f"ここを実装してください。ヒント: {hint}")

USE_MOCK_LLM = False  # 本物のモデルを試す準備ができたら False に変更してください

# --- 本物のモデルを使うときの読み込み方法(2通り、モジュール1と同じ考え方) ---
# "local":       S3からダウンロード済みのモデルをローカルフォルダから読み込む(デフォルト)
# "huggingface": Hugging Face Hubから直接ダウンロード(インターネット接続が必要)
MODEL_SOURCE = "local"  # ネットワークの都合でHugging Face経由を試したい場合は "huggingface" に変更
LOCAL_MODEL_PATH = "./models/Qwen2.5-0.5B-Instruct"

_generator_cache = {}  # 読み込んだモデルをここに保存し、2回目以降の呼び出しを高速化します

def get_generator():
    """real_llm_decide / real_llm_generate の両方で使う、モデル読み込みの共通処理です。
    MODEL_SOURCE の設定に応じて、Hugging Faceから直接、またはローカル/S3経由のどちらかで
    読み込みます(モジュール1のボーナスセルと同じ考え方です)。

    重要: モデルの読み込み(「Loading weights」)自体にも時間がかかるため、一度読み込んだ
    ら _generator_cache に保存し、次回以降は再利用します。これがないと、判断・生成のたびに
    毎回モデルを読み込み直すことになり、実質的な待ち時間が2〜3倍に膨らんでしまいます。"""
    if MODEL_SOURCE in _generator_cache:
        return _generator_cache[MODEL_SOURCE]

    import os, warnings
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # 重みのロード進捗バーを消す
    warnings.filterwarnings("ignore")  # transformersの非推奨警告などを消す

    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()  # transformers自身のログ出力も抑える

    from transformers import pipeline
    model_ref = LOCAL_MODEL_PATH if MODEL_SOURCE == "local" else "Qwen/Qwen2.5-0.5B-Instruct"
    generator = pipeline("text-generation", model=model_ref)
    # モデルに同梱されているデフォルトの generation_config が max_length=20 を
    # 設定しているため、呼び出し時に max_new_tokens を指定すると競合警告が出ます。
    # ここで max_length を無効化し、常に max_new_tokens だけが使われるようにします。
    generator.model.generation_config.max_length = None

    _generator_cache[MODEL_SOURCE] = generator
    return generator

print(f"準備完了です。fastmcp {fastmcp_version}. USE_MOCK_LLM = {USE_MOCK_LLM}")

### 本物のモデルを使う予定がある場合は、ここで取得しておいてください(任意)

このノートブックの後半にある「ボーナス」で `USE_MOCK_LLM = False` に切り替えると、
本物の指示モデル(`Qwen2.5-0.5B-Instruct`)が必要になります。**そのモデルはステップ4より
前に読み込まれる**ため、後半のボーナスまで取得を待つと、ステップ4でエラーになります。

本物のモデルを試す予定があるなら、今のうちに次のセルを実行しておいてください。今日は
`USE_MOCK_LLM = True`(デフォルト)のまま進めるなら、このセルは**実行しなくてもかまい
ません**(スキップしても後の演習に影響しません)。

In [ ]:
# このワークショップ用に、モデル一式をS3バケット(公開読み取り可能)に事前配置しています。
# --no-sign-request を付けることで、AWS認証情報なしで公開バケットから取得できます。
!aws s3 sync s3://llm-workshop-files/Qwen2.5-0.5B-Instruct ./models/Qwen2.5-0.5B-Instruct --no-sign-request

## ステップ1: モジュール1とモジュール2を持ち込む

RAGとMCPをゼロから作り直すのではなく、完成したものをそのまま再利用します(複数
ファイルのプロジェクトであれば、モジュール1・2のコードを`import`で読み込むところ
です)。

In [ ]:
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

# --- モジュール1から ---
DOCS = [
    "ニンバス・スカウトドローンの飛行時間は20分で、価格は249ドルです。",
    "ニンバス・カーゴドローンは最大5キログラムまで運搬でき、価格は899ドルです。",
    "返品は購入から30日以内で、元の梱包があれば受け付けます。",
    "ニンバス・ロボティクスの保証は、製造上の欠陥について12か月間保証します。",
    "ニンバスのドローンのバッテリーをフル充電するには約90分かかります。",
]

def chunk_documents(docs, max_chars=60):
    chunks = []
    for doc in docs:
        for i in range(0, len(doc), max_chars):
            chunks.append(doc[i:i+max_chars])
    return chunks

def embed_chunks_local(chunks):
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    vectors = vectorizer.fit_transform(chunks).toarray().astype("float32")
    return vectors, vectorizer

def retrieve(query, vectorizer, index, chunks, k=2):
    qvec = vectorizer.transform([query]).toarray().astype("float32")
    distances, indices = index.search(qvec, k)
    return [chunks[i] for i in indices[0]]

chunks = chunk_documents(DOCS)
vectors, vectorizer = embed_chunks_local(chunks)
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

# --- モジュール2から ---
INVENTORY = {
    "drone-100": {"name": "ニンバス・スカウトドローン", "stock": 12, "price": 249.00},
    "drone-200": {"name": "ニンバス・カーゴドローン", "stock": 3, "price": 899.00},
}

mcp = FastMCP("NimbusInventory")

@mcp.tool
def check_inventory(product_id: str) -> dict:
    """ニンバス・ロボティクスの製品IDを指定して、在庫数と価格を確認する。"""
    item = INVENTORY.get(product_id)
    if not item:
        return {"error": f"Unknown product_id '{product_id}'"}
    return item

print("モジュール1 + 2の部品を読み込みました。")

## ステップ2: モデルの「判断フォーマット」

モデルは、エージェントループが解釈できる形式で出力する必要があります —— 小さなJSON
スキーマを使います:`{"tool_call": "...", "args": {...}}`、
`{"retrieve": "検索クエリ"}`、または `{"response": "直接の回答"}` のいずれかです。

これはモジュール2のツールのスキーマと同じ考え方です。docstringがモデルに「このツールが
何をするか」を伝えるように、この判断フォーマットは、エージェントループに「モデルが
何をすると決めたか」を伝えます。

In [ ]:
def mock_llm_decide(user_query: str) -> str:
    """本物の指示モデルの代わりとなる模擬関数。簡単なキーワードのルールで、ツール呼び出し・
    検索・直接応答のどれにするかを決めます -- 本物のモデルに切り替える前に、エージェント
    ループの制御フローを試すのに十分な仕組みです。"""
    product_match = re.search(r"drone-\d+", user_query)
    if "在庫" in user_query and product_match:
        return json.dumps({"tool_call": "check_inventory", "args": {"product_id": product_match.group()}})
    if any(word in user_query for word in ["保証", "返品", "充電", "飛行時間"]):
        return json.dumps({"retrieve": user_query})
    return json.dumps({"response": "製品に関する質問、在庫状況、ポリシーについてお答えできます。"})

def real_llm_decide(user_query: str) -> str:
    """本物のバージョン -- transformers経由で小さな指示モデルをローカルで実行します。
    USE_MOCK_LLM = False のときだけ使われます。

    このシステムプロンプトは日本語で書かれ、3つの選択肢すべてに具体例を1つずつ
    含んでいます。当初は英語のプロンプトで tool_call の例だけが具体的だったため、
    モデルが retrieve をほとんど選ばない偏りが実際に観察されました -- 3つの選択肢に
    同じくらい具体的な例を与えることで、この偏りを減らせるかを確認します。"""
    generator = get_generator()
    system_prompt = (
        "あなたはユーザーの質問を3種類に振り分けるアシスタントです。"
        "他の文章を一切含めず、次のいずれか1つのJSONオブジェクトだけを返してください。\n\n"
        "①在庫や注文について、製品IDを使って調べる場合:\n"
        '例)「drone-100の在庫はありますか?」→ {"tool_call": "check_inventory", "args": {"product_id": "drone-100"}}\n\n'
        "②保証・返品・充電時間・飛行時間など、製品ドキュメントを検索して答える場合:\n"
        '例)「保証期間はどれくらいですか?」→ {"retrieve": "保証期間はどれくらいですか?"}\n'
        '例)「返品はできますか?」→ {"retrieve": "返品はできますか?"}\n\n'
        "③製品ドキュメントにもツールにも当てはまらない質問の場合:\n"
        '例)「ドローンの色は選べますか?」→ {"response": "その情報は今は持ち合わせていませんが、製品の在庫や保証、返品についてはお答えできます。"}'
    )
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_query}]
    output = generator(messages, max_new_tokens=60)
    return output[0]["generated_text"][-1]["content"]

llm_decide = mock_llm_decide if USE_MOCK_LLM else real_llm_decide
print("使用中:", llm_decide.__name__)

## ステップ2.5: 生成ステップ —— RAGの「AG」を完成させる

モジュール1の `retrieve()` は、生のチャンクをそのまま返します。これは便利ですが、
回答そのものではありません —— 回答の**材料**にすぎません。「拡張生成」とは、そのチャンクを
文脈としてモデルに渡し、実際に自分の言葉で質問に答えてもらうことです。

これには専用の関数 `llm_generate` が必要です。上の `llm_decide` とは別のものです ——
1回目のモデル呼び出しは「何をするか」を決め、2回目のモデル呼び出し(検索が選ばれた
ときだけ実行される)が「答える」役割を担います。実際の本番のRAGシステムでも、この
2つを分けて呼び出します。理由は同じです:判断と、根拠のある回答は別の仕事であり、
一緒くたにすると両方ともデバッグしにくくなるからです。

In [ ]:
def mock_llm_generate(user_query: str, context_chunks: list) -> str:
    """本物の指示モデルの「生成」ステップの代わりとなる模擬関数。取得した文脈を回答らしく
    言い換える簡単なテンプレートです -- 本当の生成ではありませんが、生のチャンクの
    羅列ではなく、組み立てられた回答としてエージェントが応答することを示すには十分です。"""
    context = " ".join(context_chunks)
    return f"次の情報が見つかりました: {context}"

def real_llm_generate(user_query: str, context_chunks: list) -> str:
    """本物のバージョン -- 取得したチャンクだけを文脈として使い、小さな指示モデルに
    質問へ回答させます。USE_MOCK_LLM = False のときだけ使われます。"""
    generator = get_generator()
    context = "\n".join(context_chunks)
    system_prompt = (
        "Answer the user's question using ONLY the context provided. "
        "If the answer isn't in the context, say you don't know."
    )
    user_content = f"Context:\n{context}\n\nQuestion: {user_query}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_content}]
    output = generator(messages, max_new_tokens=80)
    return output[0]["generated_text"][-1]["content"]

llm_generate = mock_llm_generate if USE_MOCK_LLM else real_llm_generate
print("使用中:", llm_generate.__name__)

## ステップ3: エージェントループ

**あなたの番:** `agent_loop` を完成させてください。ユーザーの質問を受け取ったら:
1. `llm_decide(user_query)` を呼び、結果を `json.loads` で読み込む
2. `"tool_call"` キーがあれば、MCPの `Client` でそのツールを `"args"` 付きで呼び出し、
   結果を説明する文字列を返す
3. `"retrieve"` キーがあれば、`retrieve(...)` でチャンクを取得し、
   `llm_generate(user_query, found)` を呼んで実際の回答に変える —— これがRAGの
   「R」(検索)に足りなかった「AG」(生成)を完成させるステップです
4. `"response"` キーがあれば、その文字列をそのまま返す

In [ ]:
async def agent_loop(user_query: str) -> str:
    raw = llm_decide(user_query)
    decision = json.loads(raw)

    if "tool_call" in decision:
        # TODO: MCPのClient(mcp)を使い、decision["tool_call"]をdecision["args"]付きで呼び出し、
        # 結果を説明する文字列を返してください
        TODO("async with Client(mcp) as client: decision['tool_call'] を decision['args'] で呼び出す")
    elif "retrieve" in decision:
        # TODO: decision["retrieve"]でretrieve(...)を呼んでチャンクを取得し、
        # llm_generate(user_query, found)を呼んで実際の回答に変えてください
        TODO("found = retrieve(...); return llm_generate(user_query, found)")
    else:
        return decision["response"]

In [ ]:
# --- レスキューセル ---
async def agent_loop(user_query: str) -> str:
    raw = llm_decide(user_query)
    decision = json.loads(raw)

    if "tool_call" in decision:
        async with Client(mcp) as client:
            result = await client.call_tool(decision["tool_call"], decision["args"])
            return f"ツールの結果: {result.data}"
    elif "retrieve" in decision:
        found = retrieve(decision["retrieve"], vectorizer, index, chunks)
        return llm_generate(user_query, found)
    else:
        return decision["response"]

## ステップ4: 実際に最初から最後まで試す

In [ ]:
for q in [
    "drone-200の在庫はありますか?",
    "保証期間はどれくらいですか?",
    "ドローンの色は選べますか?",
]:
    print("質問:", q)
    try:
        print("回答:", await agent_loop(q))
    except Exception as e:
        # 本物のモデルは、存在しないツール名を呼び出そうとするなど、予期しない形で
        # 失敗することがあります。1つの質問の失敗で残りの質問まで止まらないように、
        # ここでエラーを捕まえて表示します。
        print(f"エラーが発生しました: {type(e).__name__}: {e}")
    print()

## ステップ5: 失敗を観察してから、直す

これはモジュール1のまとめで触れた「信頼性」の教訓です。実際の指示モデルは、特に
小さなモデルほど、指定した出力フォーマットを毎回きちんと守るとは限りません。JSON の
周りに会話的な前置きをつけてしまうモデルをここで再現してみます(これは実際に非常に
よくある失敗パターンです)。

In [ ]:
def flaky_llm_decide(user_query: str) -> str:
    """JSONの判断結果を余計な文章で包んでしまうモデルを再現します -- このままでは
    json.loadsが直接失敗します。"""
    product_match = re.search(r"drone-\d+", user_query)
    if product_match:
        return f'かしこまりました、こちらが回答です: {{"tool_call": "check_inventory", "args": {{"product_id": "{product_match.group()}"}}}} 何かご不明な点があればどうぞ!'
    return '{"response": "製品に関するご質問にお答えします。"}'

llm_decide = flaky_llm_decide
try:
    print(await agent_loop("drone-100の在庫はありますか?"))
except Exception as e:
    print(f"予想通り失敗しました: {type(e).__name__}: {e}")

**あなたの番:** `agent_loop`(またはそこから呼び出す補助関数)を修正して、モデルが
余計な文章でJSONを包んでいても、正規表現で最初の `{...}` ブロックを見つけてから
`json.loads` を呼べるようにしてください。これは、実際のエージェントフレームワークでも
使われている技術です(「JSON抽出」や「出力パース」と呼ばれます)。

In [ ]:
def extract_json(raw: str) -> dict:
    # TODO: re.searchを使ってrawの中から最初の{...}ブロックを見つけ、json.loadsしてください
    TODO("re.search(r'\\{.*\\}', raw, re.DOTALL) でマッチさせ、そのグループをjson.loadsする")


In [ ]:
# --- レスキューセル ---
def extract_json(raw: str) -> dict:
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in model output: {raw!r}")
    return json.loads(match.group())

async def agent_loop(user_query: str) -> str:
    raw = llm_decide(user_query)
    decision = extract_json(raw)

    if "tool_call" in decision:
        async with Client(mcp) as client:
            result = await client.call_tool(decision["tool_call"], decision["args"])
            return f"ツールの結果: {result.data}"
    elif "retrieve" in decision:
        found = retrieve(decision["retrieve"], vectorizer, index, chunks)
        return llm_generate(user_query, found)
    else:
        return decision["response"]

print(await agent_loop("drone-100の在庫はありますか?"))

## ボーナス(任意・インターネット接続と数分が必要): 本物のモデルを使う

フラグを切り替えて、ステップ2から再実行し、`Qwen2.5-0.5B-Instruct` を本物で使って
みましょう。同じ質問を試して比較してください: 本物の小さなモデルは、模擬関数と同じくらい
JSONフォーマットを守れるでしょうか? システムプロンプトの例にない質問をしたら、どうなる
でしょうか?

```python
USE_MOCK_LLM = False
```

> ⚠️ **カーネル再起動について:** モジュール1と同様、初めて重いライブラリを読み込む際に
> Jupyterがカーネルの再起動を求めることがあります。求められたら、慌てずに「Restart
> Kernel and Run All Cells」を選んで最初から再実行してください。

> ⚠️ **重要:** `USE_MOCK_LLM = False` に切り替えたら、**ステップ1より前の、セットアップ
> セクションにあるモデル取得セルをすでに実行済みであることを確認してから**
> ステップ2以降を再実行してください。モデルがまだローカルに存在しない状態で
> `USE_MOCK_LLM = False` のままステップ4まで進むと、エラーになります。

## RAGとMCP、それぞれを確認するための質問例

本物のモデルが実際にRAG(`retrieve`)とMCPツール(`tool_call`)の両方を正しく
使い分けられているかを確認するために、次の質問を試してみてください。

**① MCPツール(`tool_call`)が選ばれるはずの質問:**
- `drone-100の在庫はありますか?`
- `drone-200は何個ありますか?`

**② RAG(`retrieve`)が選ばれるはずの質問(製品ドキュメントに書かれている内容):**
- `保証期間はどれくらいですか?`
- `返品はできますか?`
- `充電にはどれくらい時間がかかりますか?`
- `飛行時間はどれくらいですか?`

**③ 直接回答(`response`)が選ばれるはずの質問(ツールにもドキュメントにも
ない内容):**
- `ドローンの色は選べますか?`
- `営業時間を教えてください`

> 📌 **確認のポイント:** ①②③のうち①③はシステムプロンプトに書いた例と似た文言なので、
> ある程度は正しく判断できて当然です。**本当に大事なのは②が確実に動くかどうかです**
> —— これがRAGが実際に呼び出される経路だからです。もし②の質問が `tool_call` や
> `response` に誤って振り分けられたら、それはRAGが壊れているのではなく、`llm_decide`
> の判断がRAGへの経路を選べていない、ということです(このモジュールの前半で確認した
> 区別と同じです)。

> 💭 **さらに試してみましょう:** 上のリストにない、似たような言い回しの質問も作って
> みてください(例: `保証はどのくらい続きますか?`)。プロンプトに書いた例文と完全に
> 一致しない質問でも正しく振り分けられるなら、モデルが単に例文を暗記しているのではなく、
> 本当に「意図」を理解して判断していると言えます。

> ⚠️ **同じ質問でも、実行するたびに結果が変わることがあります:** `Qwen2.5-0.5B-Instruct`
> は、出力を生成する際にある程度のランダム性(サンプリング)を使っています。そのため、
> **まったく同じ質問を2回実行しても、毎回同じ判断結果になるとは限りません。** ある回では
> 正しく`retrieve`が選ばれても、別の回では間違った`tool_call`が選ばれることがあります。
> 隣の人と同じ質問を試して、違う結果になったとしても、それはどちらかが操作を間違えた
> わけではなく、この非決定性が原因である可能性が高いです。

## ボーナス+: 自分で質問を入力できる対話ループ

`ipywidgets`のボタンUIは、非同期処理(`asyncio`)とウィジェットのイベント処理が
組み合わさることで、原因の特定が難しい不具合(タスクが完了せず、エラーも出ないまま
応答が返ってこない)が起きたため、**もっと単純な方法**に切り替えます。

代わりに、Pythonの`input()`を使った対話ループにします。ステップ4とまったく同じ
`await agent_loop(q)`の呼び出し方をそのまま繰り返すだけなので、ステップ4で実際に
動作確認できている経路と完全に同じです。

使い方: 下のセルを実行すると、質問を聞かれます。入力して Enter を押してください。
`終了` と入力するとループを抜けます。

In [ ]:
# ステップ5で llm_decide は意図的に flaky_llm_decide に置き換えられたままです。
# ここで、現在の USE_MOCK_LLM の設定に応じた正しい関数に戻します。
llm_decide = mock_llm_decide if USE_MOCK_LLM else real_llm_decide

async def interactive_qa():
    print("質問を入力してください(終了するには「終了」と入力してEnter)")
    while True:
        q = input("質問: ")
        if q.strip() in ("終了", "quit", "exit", ""):
            print("終了します。")
            break
        try:
            print("回答:", await agent_loop(q))
        except Exception as e:
            # 本物のモデルは、存在しないツール名を呼び出そうとするなど、予期しない形で
            # 失敗することがあります(ステップ4と同じ理由です)。
            print(f"エラーが発生しました: {type(e).__name__}: {e}")
        print()

await interactive_qa()

## まとめ

判断・検索・ツール呼び出し・生成を1つにまとめたループが完成しました。検索が選ばれた
ときは、2回目のモデル呼び出しが生のチャンクを実際の回答に変え、ワークショップの中で
初めてRAGの「AG」(拡張生成)を完成させました。また、実際のエージェントフレームワークと
同じ方法で、現実的な失敗(不正な形式のJSON出力)にも対処しました。Day 2午前では、
この手作りのループを、同じRAGの索引とMCPサーバーを使いながら、AWSの管理された
エージェントフレームワークと比較します。